# Adding Large Inductive Datasets with On-Disk Processing

This tutorial shows how to add and use large inductive datasets (many graphs) that exceed RAM capacity using TopoBench's on-disk infrastructure.

## Why On-Disk?

Traditional in-memory preprocessing loads ALL structures (triangles, cliques) into RAM:
- **Problem**: Large datasets → millions of structures → RAM exhaustion (OOM)
- **Solution**: On-disk streaming enumeration → constant memory (~50-100MB)

**Use on-disk when:**
- Dataset has many graphs (>1000)
- Graphs are large (>50 nodes)
- High degree graphs (many triangles)
- Limited RAM (<8GB)

## Step 1: Create Your Dataset Class

Follow TopoBench pattern - inherit from `InMemoryDataset`:

In [ ]:
import networkx as nx
import torch
from torch_geometric.data import Data, InMemoryDataset
from torch_geometric.io import fs
from omegaconf import DictConfig
import os.path as osp

class MyLargeInductiveDataset(InMemoryDataset):
    """Your custom large inductive dataset."""
    
    def __init__(self, root, name, parameters: DictConfig):
        self.name = name
        self.parameters = parameters
        super().__init__(root)
        
        # Load processed data
        out = fs.torch_load(self.processed_paths[0])
        if len(out) == 4:
            data, self.slices, self.sizes, data_cls = out
            self.data = data_cls.from_dict(data) if isinstance(data, dict) else data
        else:
            data, self.slices, self.sizes = out
            self.data = data
    
    @property
    def raw_file_names(self):
        return []  # Implement if downloading
    
    @property
    def processed_file_names(self):
        return "data.pt"
    
    def download(self):
        pass  # Implement your download logic
    
    def process(self):
        """Generate/load your graphs here."""
        data_list = []
        
        # Example: Generate synthetic graphs
        for i in range(self.parameters.num_graphs):
            G = nx.watts_strogatz_graph(
                n=self.parameters.nodes_per_graph,
                k=self.parameters.degree,
                p=0.3,
                seed=42+i
            )
            
            # Convert to PyG Data
            edges = list(G.edges())
            edge_index = torch.tensor(edges, dtype=torch.long).t()
            edge_index = torch.cat([edge_index, edge_index[[1, 0]]], dim=1)
            
            x = torch.randn(G.number_of_nodes(), self.parameters.num_features)
            y = torch.randint(0, self.parameters.num_classes, (1,))
            
            data = Data(x=x, edge_index=edge_index, y=y)
            data_list.append(data)
        
        # Collate and save
        self.data, self.slices = self.collate(data_list)
        fs.torch_save(
            (self._data.to_dict(), self.slices, {}, self._data.__class__),
            self.processed_paths[0]
        )

## Step 2: Create Your Loader

Inherit from `AbstractLoader`:

In [ ]:
from topobench.data.loaders.base import AbstractLoader
from pathlib import Path

class MyLargeInductiveLoader(AbstractLoader):
    """Loader for your custom dataset."""
    
    def __init__(self, parameters: DictConfig):
        super().__init__(parameters)
    
    def load_dataset(self):
        dataset = MyLargeInductiveDataset(
            root=str(self.root_data_dir),
            name=self.parameters.data_name,
            parameters=self.parameters
        )
        return dataset

## Step 3: Use On-Disk Processing

Instead of `PreProcessor`, use `OnDiskInductiveDataset`:

In [ ]:
from omegaconf import OmegaConf
from topobench.data.preprocessor.ondisk_inductive import OnDiskInductiveDataset

# Configure your dataset
loader_config = OmegaConf.create({
    "data_dir": "./data/",
    "data_name": "MyLargeDataset",
    "num_graphs": 5000,
    "nodes_per_graph": 80,
    "degree": 15,
    "num_features": 16,
    "num_classes": 5
})

# Load dataset
loader = MyLargeInductiveLoader(loader_config)
dataset, dataset_dir = loader.load()

print(f"Loaded {len(dataset)} graphs")

# Use ON-DISK preprocessing (constant memory!)
ondisk_dataset = OnDiskInductiveDataset(
    dataset=dataset,
    data_dir="./tmp/ondisk_index",
    max_k=2,  # Include up to triangles
    force_rebuild=False
)

print("✓ On-disk preprocessing complete (constant memory)")

## Step 4: Create Splits and Dataloader

In [ ]:
from topobench.dataloader import TBDataloader

# Create splits
n_train = int(0.5 * len(ondisk_dataset))
n_val = int(0.25 * len(ondisk_dataset))

indices = torch.randperm(len(ondisk_dataset)).tolist()
train_indices = indices[:n_train]
val_indices = indices[n_train:n_train+n_val]
test_indices = indices[n_train+n_val:]

dataset_train = torch.utils.data.Subset(ondisk_dataset, train_indices)
dataset_val = torch.utils.data.Subset(ondisk_dataset, val_indices)
dataset_test = torch.utils.data.Subset(ondisk_dataset, test_indices)

# Create dataloader
datamodule = TBDataloader(dataset_train, dataset_val, dataset_test, batch_size=32)

## Step 5: Train Your Model

In [ ]:
import pytorch_lightning as pl
from topobench.nn.encoders import AllCellFeatureEncoder
from topobench.nn.backbones.simplicial.sccnn import SCCNNCustom
from topobench.nn.wrappers import SCCNNWrapper
from topobench.nn.readouts import PropagateSignalDown
from topobench.loss import TBLoss
from topobench.evaluator import TBEvaluator
from topobench.optimizer import TBOptimizer
from topobench.model import TBModel

# Setup model
HIDDEN_DIM = 32
OUT_CHANNELS = 5

feature_encoder = AllCellFeatureEncoder(
    in_channels=[16, 16, 16],
    out_channels=HIDDEN_DIM,
    selected_dimensions=[0, 1, 2]
)

backbone = SCCNNCustom(
    in_channels_all=[HIDDEN_DIM]*3,
    hidden_channels_all=[HIDDEN_DIM]*3,
    conv_order=1,
    sc_order=3,
    n_layers=1
)

backbone_wrapper = SCCNNWrapper(backbone, out_channels=HIDDEN_DIM, num_cell_dimensions=3)

readout = PropagateSignalDown(
    num_cell_dimensions=3,
    hidden_dim=HIDDEN_DIM,
    out_channels=OUT_CHANNELS,
    task_level="graph",
    pooling_type="sum"
)

loss = TBLoss(OmegaConf.create({"dataset_loss": {"task": "classification", "loss_type": "cross_entropy"}}))
evaluator = TBEvaluator(OmegaConf.create({"task": "classification", "num_classes": OUT_CHANNELS, "metrics": ["accuracy"]}))
optimizer = TBOptimizer(OmegaConf.create({"optimizer_id": "Adam", "parameters": {"lr": 0.01}}))

model = TBModel(
    backbone=backbone_wrapper,
    readout=readout,
    loss=loss,
    feature_encoder=feature_encoder,
    evaluator=evaluator,
    optimizer=optimizer
)

# Train
trainer = pl.Trainer(max_epochs=10, accelerator="cpu")
trainer.fit(model, datamodule)

print("✅ Training complete with on-disk dataset!")

## Key Differences: In-Memory vs On-Disk

| Aspect | In-Memory (`PreProcessor`) | On-Disk (`OnDiskInductiveDataset`) |
|--------|---------------------------|------------------------------------|
| **Memory** | Grows with dataset size | Constant (~50-100MB) |
| **Preprocessing** | All structures in RAM | Streams to disk |
| **Disk Usage** | Minimal | Uses disk for index |
| **Speed** | Fast (all in RAM) | Slightly slower (disk I/O) |
| **Scalability** | Limited by RAM | Limited by disk |
| **Use When** | Small datasets | Large datasets |

## Summary

1. **Create dataset class** (inherit `InMemoryDataset`)
2. **Create loader** (inherit `AbstractLoader`)
3. **Use `OnDiskInductiveDataset`** instead of `PreProcessor`
4. **Train normally** - rest of workflow unchanged!

**Result**: Train on datasets that would OOM with in-memory approach! 🎊